# Phase 1A Deliverables 1-8 Validation

This notebook is a human-readable acceptance check for Phase 1A. It does not replace pytest coverage; it gives visible spot checks and artifact consistency checks for Deliverables 1-8:

- Deliverable 1: `backtest/bt_parquet_feed.py`
- Deliverable 2: `backtest/execution_model.py` and `backtest/slippage.py`
- Deliverables 3-8: engine, metrics, registry, strategy template, SMA baseline, and trade audit acceptance on a real run

Run the notebook top to bottom. A failed deliverable check raises an assertion error in the cell where the mismatch is found.

## Setup

In [1]:
from pathlib import Path
import os
import sys
import tempfile

import backtrader as bt
import numpy as np
import pandas as pd
from IPython.display import display



def _bootstrap_repo_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "backtest").exists():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            os.chdir(candidate)
            return candidate
    raise RuntimeError(f"Could not locate repo root from {start}")


PROJECT_ROOT = _bootstrap_repo_root()
print("Notebook repo root:", PROJECT_ROOT)

from backtest.bt_parquet_feed import ParquetFeed
from backtest.execution_model import AlphaBroker, configure_cerebro

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)

PARQUET_PATH = Path("data/raw/btcusdt_1h.parquet")
COMMISSION_RATE = 0.0007

assert PARQUET_PATH.exists(), f"Missing canonical parquet: {PARQUET_PATH}"
print("Validation target:", PARQUET_PATH)
print("Expected commission rate:", COMMISSION_RATE)


Validation target: data/raw/btcusdt_1h.parquet
Expected commission rate: 0.0007


## Part A - Deliverable 1: ParquetFeed

The feed passes this section if:

- Backtrader sees the same row count as pandas.
- Backtrader datetimes align one-for-one with `open_time_utc` after stripping UTC tzinfo for Backtrader.
- OHLCV values match the direct pandas read.
- `fromdate` and `todate` filtering returns the same inclusive range as pandas.

### A1. Direct pandas read of the canonical parquet

In [2]:
raw_df = (
    pd.read_parquet(PARQUET_PATH)
    .sort_values("open_time_utc")
    .reset_index(drop=True)
)

required_cols = ["open_time_utc", "open", "high", "low", "close", "volume"]
missing_cols = set(required_cols) - set(raw_df.columns)
assert not missing_cols, f"Canonical parquet missing columns: {sorted(missing_cols)}"

print("Rows:", len(raw_df))
print("Start:", raw_df["open_time_utc"].min())
print("End:", raw_df["open_time_utc"].max())
display(raw_df.head(5)[required_cols])


Rows: 55105
Start: 2020-01-01 00:00:00+00:00
End: 2026-04-16 07:00:00+00:00


,open_time_utc,open,high,low,close,volume
0,2020-01-01 00:00:00+00:00,7195.24,7196.25,7175.46,7177.02,511.814901
1,2020-01-01 01:00:00+00:00,7176.47,7230.00,7175.71,7216.27,883.052603
2,2020-01-01 02:00:00+00:00,7215.52,7244.87,7211.41,7242.85,655.156809
3,2020-01-01 03:00:00+00:00,7242.66,7245.00,7220.00,7225.01,783.724867
4,2020-01-01 04:00:00+00:00,7225.00,7230.00,7215.03,7217.27,467.812578


### A2. Full feed alignment against pandas

In [3]:
class CollectFeedStrategy(bt.Strategy):
    def __init__(self):
        self.rows = []

    def next(self):
        self.rows.append({
            "dt": self.data.datetime.datetime(0),
            "open": float(self.data.open[0]),
            "high": float(self.data.high[0]),
            "low": float(self.data.low[0]),
            "close": float(self.data.close[0]),
            "volume": float(self.data.volume[0]),
        })


def run_feed_collection(feed):
    cerebro = bt.Cerebro()
    cerebro.adddata(feed)
    cerebro.addstrategy(CollectFeedStrategy)
    return cerebro.run()[0]

feed = ParquetFeed.from_parquet(PARQUET_PATH)
collector = run_feed_collection(feed)
feed_df = pd.DataFrame(collector.rows)

expected_df = raw_df[required_cols].copy()
expected_dt_naive = expected_df["open_time_utc"].dt.tz_localize(None).reset_index(drop=True)

assert len(feed_df) == len(raw_df), f"Row count mismatch: feed={len(feed_df)}, pandas={len(raw_df)}"
pd.testing.assert_series_equal(feed_df["dt"].reset_index(drop=True), expected_dt_naive, check_names=False, check_dtype=False)

for col in ["open", "high", "low", "close", "volume"]:
    np.testing.assert_allclose(feed_df[col].to_numpy(), expected_df[col].to_numpy(), rtol=0, atol=1e-12)

comparison = pd.concat(
    [
        expected_df.head(5).rename(columns={"open_time_utc": "pandas_dt"}).reset_index(drop=True),
        feed_df.head(5).add_prefix("feed_").reset_index(drop=True),
    ],
    axis=1,
)

display(comparison)
print("PASS: Deliverable 1 full-feed row count, datetime alignment, and OHLCV values match pandas.")


,pandas_dt,open,high,low,close,volume,feed_dt,feed_open,feed_high,feed_low,feed_close,feed_volume
0,2020-01-01 00:00:00+00:00,7195.24,7196.25,7175.46,7177.02,511.814901,2020-01-01 00:00:00,7195.24,7196.25,7175.46,7177.02,511.814901
1,2020-01-01 01:00:00+00:00,7176.47,7230.00,7175.71,7216.27,883.052603,2020-01-01 01:00:00,7176.47,7230.00,7175.71,7216.27,883.052603
2,2020-01-01 02:00:00+00:00,7215.52,7244.87,7211.41,7242.85,655.156809,2020-01-01 02:00:00,7215.52,7244.87,7211.41,7242.85,655.156809
3,2020-01-01 03:00:00+00:00,7242.66,7245.00,7220.00,7225.01,783.724867,2020-01-01 03:00:00,7242.66,7245.00,7220.00,7225.01,783.724867
4,2020-01-01 04:00:00+00:00,7225.00,7230.00,7215.03,7217.27,467.812578,2020-01-01 04:00:00,7225.00,7230.00,7215.03,7217.27,467.812578


PASS: Deliverable 1 full-feed row count, datetime alignment, and OHLCV values match pandas.


### A2b. Extra Backtrader lines: quote volume and trade count

`ParquetFeed` exposes `quote_volume` and `trade_count` as custom Backtrader lines. This spot-check confirms those extra lines match the direct pandas read for the first five bars.

In [4]:
extra_cols = ["quote_volume", "trade_count"]
missing_extra_cols = set(extra_cols) - set(raw_df.columns)
assert not missing_extra_cols, f"Canonical parquet missing extra feed columns: {sorted(missing_extra_cols)}"


class CollectExtraLinesStrategy(bt.Strategy):
    def __init__(self):
        self.rows = []

    def next(self):
        if len(self.rows) < 5:
            self.rows.append({
                "dt": self.data.datetime.datetime(0),
                "quote_volume": float(self.data.quote_volume[0]),
                "trade_count": float(self.data.trade_count[0]),
            })


feed = ParquetFeed.from_parquet(PARQUET_PATH)
cerebro = bt.Cerebro()
cerebro.adddata(feed)
cerebro.addstrategy(CollectExtraLinesStrategy)
extra_strat = cerebro.run()[0]
extra_feed_df = pd.DataFrame(extra_strat.rows)

expected_extra_df = raw_df[["open_time_utc", "quote_volume", "trade_count"]].head(5).copy()
expected_extra_dt_naive = expected_extra_df["open_time_utc"].dt.tz_localize(None).reset_index(drop=True)

pd.testing.assert_series_equal(extra_feed_df["dt"].reset_index(drop=True), expected_extra_dt_naive, check_names=False, check_dtype=False)
np.testing.assert_allclose(extra_feed_df["quote_volume"].to_numpy(), expected_extra_df["quote_volume"].to_numpy(), rtol=0, atol=1e-12)
np.testing.assert_allclose(extra_feed_df["trade_count"].to_numpy(), expected_extra_df["trade_count"].astype(float).to_numpy(), rtol=0, atol=1e-12)

extra_comparison = pd.concat(
    [
        expected_extra_df.rename(columns={"open_time_utc": "pandas_dt"}).reset_index(drop=True),
        extra_feed_df.add_prefix("feed_").reset_index(drop=True),
    ],
    axis=1,
)

display(extra_comparison)
print("PASS: Deliverable 1 extra lines quote_volume and trade_count match pandas for the first five bars.")


,pandas_dt,quote_volume,trade_count,feed_dt,feed_quote_volume,feed_trade_count
0,2020-01-01 00:00:00+00:00,3.675857e+06,7640,2020-01-01 00:00:00,3.675857e+06,7640.0
1,2020-01-01 01:00:00+00:00,6.365953e+06,9033,2020-01-01 01:00:00,6.365953e+06,9033.0
2,2020-01-01 02:00:00+00:00,4.736719e+06,7466,2020-01-01 02:00:00,4.736719e+06,7466.0
3,2020-01-01 03:00:00+00:00,5.667367e+06,8337,2020-01-01 03:00:00,5.667367e+06,8337.0
4,2020-01-01 04:00:00+00:00,3.379094e+06,5896,2020-01-01 04:00:00,3.379094e+06,5896.0


PASS: Deliverable 1 extra lines quote_volume and trade_count match pandas for the first five bars.


### A3. `fromdate` / `todate` slicing

In [5]:
start_utc = pd.Timestamp("2024-01-01 00:00:00", tz="UTC")
end_utc = pd.Timestamp("2024-01-03 23:00:00", tz="UTC")

expected_slice = raw_df[
    (raw_df["open_time_utc"] >= start_utc) &
    (raw_df["open_time_utc"] <= end_utc)
].reset_index(drop=True)

assert len(expected_slice) > 0, "Expected the canonical parquet to contain the validation date range"
expected_first = expected_slice["open_time_utc"].iloc[0].tz_localize(None)
expected_last = expected_slice["open_time_utc"].iloc[-1].tz_localize(None)

range_cases = [
    (
        "naive datetime interpreted as UTC",
        start_utc.to_pydatetime().replace(tzinfo=None),
        end_utc.to_pydatetime().replace(tzinfo=None),
    ),
    (
        "UTC-aware datetime",
        start_utc.to_pydatetime(),
        end_utc.to_pydatetime(),
    ),
]

slice_summaries = []
case_frames = []
for label, from_arg, to_arg in range_cases:
    feed = ParquetFeed.from_parquet(PARQUET_PATH, fromdate=from_arg, todate=to_arg)
    collector = run_feed_collection(feed)
    slice_feed_df = pd.DataFrame(collector.rows)
    case_frames.append(slice_feed_df)

    assert len(slice_feed_df) == len(expected_slice), (
        f"{label} count mismatch: feed={len(slice_feed_df)}, pandas={len(expected_slice)}"
    )
    assert slice_feed_df["dt"].iloc[0] == expected_first
    assert slice_feed_df["dt"].iloc[-1] == expected_last

    for col in ["open", "high", "low", "close", "volume"]:
        np.testing.assert_allclose(slice_feed_df[col].to_numpy(), expected_slice[col].to_numpy(), rtol=0, atol=1e-12)

    slice_summaries.append({
        "case": label,
        "from_arg_type": type(from_arg).__name__,
        "from_arg_tzinfo": str(getattr(from_arg, "tzinfo", None)),
        "to_arg_tzinfo": str(getattr(to_arg, "tzinfo", None)),
        "count": len(slice_feed_df),
        "first_dt": slice_feed_df["dt"].iloc[0],
        "last_dt": slice_feed_df["dt"].iloc[-1],
    })

pd.testing.assert_frame_equal(case_frames[0], case_frames[1], check_dtype=False)

display(pd.DataFrame(slice_summaries))
print("Expected count:", len(expected_slice))
print("Expected first:", expected_slice["open_time_utc"].iloc[0])
print("Expected last:", expected_slice["open_time_utc"].iloc[-1])
print("PASS: Deliverable 1 accepts both naive and UTC-aware fromdate/todate, with identical inclusive slicing.")


,case,from_arg_type,from_arg_tzinfo,to_arg_tzinfo,count,first_dt,last_dt
0,naive datetime interpreted as UTC,datetime,None,None,72,2024-01-01,2024-01-03 23:00:00
1,UTC-aware datetime,datetime,UTC,UTC,72,2024-01-01,2024-01-03 23:00:00


Expected count: 72
Expected first: 2024-01-01 00:00:00+00:00
Expected last: 2024-01-03 23:00:00+00:00
PASS: Deliverable 1 accepts both naive and UTC-aware fromdate/todate, with identical inclusive slicing.


## Part B - Deliverable 2: execution model and slippage

This section uses deterministic toy data so the correct signal bar, fill bar, price, and commission can be inspected directly.

The execution model passes this section if:

- Buy and sell orders fill at the next bar open, not the signal bar close.
- Commission is exactly 7 bps of absolute trade value on each side.
- The broker is `AlphaBroker` and the cost model label is `effective_7bps_per_side`.
- A zero-volume fill bar defers execution to the next non-zero-volume bar.
- More than 24 consecutive zero-volume fill attempts cancel the order.

### B1. Toy data and strategy helpers

In [6]:
tmpdir_ctx = tempfile.TemporaryDirectory()
TMPDIR = Path(tmpdir_ctx.name)
print("Temporary validation parquet directory:", TMPDIR)


def make_toy_df(n_bars=8, volumes=None):
    if volumes is None:
        volumes = [10.0] * n_bars
    assert len(volumes) == n_bars
    return pd.DataFrame({
        "open_time_utc": pd.date_range("2024-01-01 00:00:00", periods=n_bars, freq="h", tz="UTC").astype("datetime64[ms, UTC]"),
        "open": [100.0 + i for i in range(n_bars)],
        "high": [101.0 + i for i in range(n_bars)],
        "low": [99.0 + i for i in range(n_bars)],
        "close": [100.0 + i for i in range(n_bars)],
        "volume": volumes,
        "quote_volume": [1000.0] * n_bars,
        "trade_count": [10] * n_bars,
        "source": pd.array(["binance_vision"] * n_bars, dtype="string"),
        "ingested_at_utc": pd.Timestamp("2026-01-01 00:00:00", tz="UTC").floor("ms"),
    })


def write_toy_parquet(df, name):
    path = TMPDIR / name
    out = df.copy()
    out["ingested_at_utc"] = out["ingested_at_utc"].astype("datetime64[ms, UTC]")
    out.to_parquet(path, engine="pyarrow", index=False)
    return path


def expected_commission(size, price):
    return abs(size) * price * COMMISSION_RATE


class BuySellOnBars(bt.Strategy):
    params = (
        ("buy_bar", 3),
        ("sell_bar", 5),
    )

    def __init__(self):
        self.bar_log = []
        self.signals = []
        self.fills = []
        self.cancelled = []

    def next(self):
        bar_num = len(self)  # 1-based Backtrader bar number
        dt = self.data.datetime.datetime(0)
        self.bar_log.append({
            "bar_num": bar_num,
            "dt": dt,
            "open": float(self.data.open[0]),
            "close": float(self.data.close[0]),
            "volume": float(self.data.volume[0]),
            "position": float(self.position.size),
        })

        if bar_num == self.p.buy_bar and not self.position:
            self.signals.append({"side": "buy", "bar_num": bar_num, "dt": dt, "close": float(self.data.close[0])})
            self.buy(size=1)
        elif bar_num == self.p.sell_bar and self.position:
            self.signals.append({"side": "sell", "bar_num": bar_num, "dt": dt, "close": float(self.data.close[0])})
            self.sell(size=1)

    def notify_order(self, order):
        if order.status == order.Completed:
            self.fills.append({
                "status": order.getstatusname(),
                "side": "buy" if order.isbuy() else "sell",
                "executed_dt": bt.num2date(order.executed.dt).replace(tzinfo=None),
                "price": float(order.executed.price),
                "size": float(order.executed.size),
                "value": float(order.executed.value),
                "comm": float(order.executed.comm),
            })
        elif order.status == order.Canceled:
            self.cancelled.append({
                "status": order.getstatusname(),
                "side": "buy" if order.isbuy() else "sell",
                "ref": order.ref,
            })


def run_execution_check(path, strategy=BuySellOnBars, **strategy_kwargs):
    feed = ParquetFeed.from_parquet(path)
    cerebro = bt.Cerebro()
    cost_model = configure_cerebro(cerebro)
    assert isinstance(cerebro.broker, AlphaBroker)
    assert cost_model.fee_model_label == "effective_7bps_per_side"
    assert cost_model.effective_commission == COMMISSION_RATE
    cerebro.adddata(feed)
    cerebro.addstrategy(strategy, **strategy_kwargs)
    strat = cerebro.run()[0]
    return strat, cost_model

toy = make_toy_df()
toy_path = write_toy_parquet(toy, "toy_validation.parquet")
display(toy[["open_time_utc", "open", "high", "low", "close", "volume"]])


Temporary validation parquet directory: /var/folders/rq/573stz453md6gpcyqpz5y_dm0000gn/T/tmphs49eeo5


,open_time_utc,open,high,low,close,volume
0,2024-01-01 00:00:00+00:00,100.0,101.0,99.0,100.0,10.0
1,2024-01-01 01:00:00+00:00,101.0,102.0,100.0,101.0,10.0
2,2024-01-01 02:00:00+00:00,102.0,103.0,101.0,102.0,10.0
3,2024-01-01 03:00:00+00:00,103.0,104.0,102.0,103.0,10.0
4,2024-01-01 04:00:00+00:00,104.0,105.0,103.0,104.0,10.0
5,2024-01-01 05:00:00+00:00,105.0,106.0,104.0,105.0,10.0
6,2024-01-01 06:00:00+00:00,106.0,107.0,105.0,106.0,10.0
7,2024-01-01 07:00:00+00:00,107.0,108.0,106.0,107.0,10.0


### B2. Next-bar-open fill and 7 bps commission

In [7]:
strat, cost_model = run_execution_check(toy_path)

signals_df = pd.DataFrame(strat.signals)
fills_df = pd.DataFrame(strat.fills)
bar_log_df = pd.DataFrame(strat.bar_log)

display(signals_df)
display(fills_df)

assert len(strat.signals) == 2, f"Expected buy and sell signals, got {len(strat.signals)}"
assert len(strat.fills) == 2, f"Expected buy and sell fills, got {len(strat.fills)}"

buy_signal = strat.signals[0]
sell_signal = strat.signals[1]
buy_fill = strat.fills[0]
sell_fill = strat.fills[1]

expected_buy_dt = toy.loc[3, "open_time_utc"].tz_localize(None)
expected_sell_dt = toy.loc[5, "open_time_utc"].tz_localize(None)
expected_buy_open = toy.loc[3, "open"]
expected_sell_open = toy.loc[5, "open"]
expected_buy_comm = expected_commission(buy_fill["size"], buy_fill["price"])
expected_sell_comm = expected_commission(sell_fill["size"], sell_fill["price"])

assert buy_signal["side"] == "buy"
assert buy_signal["bar_num"] == 3
assert buy_fill["side"] == "buy"
assert buy_fill["executed_dt"] == expected_buy_dt
assert buy_fill["price"] == expected_buy_open
assert buy_fill["price"] != buy_signal["close"], "Buy filled at signal close instead of next bar open"
assert buy_fill["comm"] == expected_buy_comm

assert sell_signal["side"] == "sell"
assert sell_signal["bar_num"] == 5
assert sell_fill["side"] == "sell"
assert sell_fill["executed_dt"] == expected_sell_dt
assert sell_fill["price"] == expected_sell_open
assert sell_fill["price"] != sell_signal["close"], "Sell filled at signal close instead of next bar open"
assert sell_fill["comm"] == expected_sell_comm

b2_checks = pd.DataFrame([
    {"check": "buy fill datetime", "expected": expected_buy_dt, "actual": buy_fill["executed_dt"], "pass": buy_fill["executed_dt"] == expected_buy_dt},
    {"check": "buy fill open", "expected": expected_buy_open, "actual": buy_fill["price"], "pass": buy_fill["price"] == expected_buy_open},
    {"check": "buy commission", "expected": expected_buy_comm, "actual": buy_fill["comm"], "pass": buy_fill["comm"] == expected_buy_comm},
    {"check": "sell fill datetime", "expected": expected_sell_dt, "actual": sell_fill["executed_dt"], "pass": sell_fill["executed_dt"] == expected_sell_dt},
    {"check": "sell fill open", "expected": expected_sell_open, "actual": sell_fill["price"], "pass": sell_fill["price"] == expected_sell_open},
    {"check": "sell commission", "expected": expected_sell_comm, "actual": sell_fill["comm"], "pass": sell_fill["comm"] == expected_sell_comm},
])
display(b2_checks)

print(f"Expected buy fill open: {expected_buy_open}; actual buy fill price: {buy_fill['price']}")
print(f"Expected buy commission: {expected_buy_comm}; actual buy commission: {buy_fill['comm']}")
print(f"Expected sell fill open: {expected_sell_open}; actual sell fill price: {sell_fill['price']}")
print(f"Expected sell commission: {expected_sell_comm}; actual sell commission: {sell_fill['comm']}")
print("PASS: Deliverable 2 next-bar-open execution and 7bps commission validated.")


,side,bar_num,dt,close
0,buy,3,2024-01-01 02:00:00,102.0
1,sell,5,2024-01-01 04:00:00,104.0


,status,side,executed_dt,price,size,value,comm
0,Completed,buy,2024-01-01 03:00:00,103.0,1.0,103.0,0.0721
1,Completed,sell,2024-01-01 05:00:00,105.0,-1.0,103.0,0.0735


,check,expected,actual,pass
0,buy fill datetime,2024-01-01 03:00:00,2024-01-01 03:00:00,True
1,buy fill open,103.0,103.0,True
2,buy commission,0.0721,0.0721,True
3,sell fill datetime,2024-01-01 05:00:00,2024-01-01 05:00:00,True
4,sell fill open,105.0,105.0,True
5,sell commission,0.0735,0.0735,True


Expected buy fill open: 103.0; actual buy fill price: 103.0
Expected buy commission: 0.0721; actual buy commission: 0.0721
Expected sell fill open: 105.0; actual sell fill price: 105.0
Expected sell commission: 0.0735; actual sell commission: 0.0735
PASS: Deliverable 2 next-bar-open execution and 7bps commission validated.


### B3. Zero-volume fill defers to the next valid bar

In [8]:
toy_zv = toy.copy()
toy_zv.loc[3, "volume"] = 0.0  # The normal buy fill bar is zero-volume.
toy_zv_path = write_toy_parquet(toy_zv, "toy_zero_volume.parquet")

display(toy_zv[["open_time_utc", "open", "close", "volume"]])

zv_strat, _ = run_execution_check(toy_zv_path)
zv_signals_df = pd.DataFrame(zv_strat.signals)
zv_fills_df = pd.DataFrame(zv_strat.fills)

display(zv_signals_df)
display(zv_fills_df)

assert len(zv_strat.fills) == 2, f"Expected deferred buy plus later sell, got {len(zv_strat.fills)} fills"
zv_buy_fill = zv_strat.fills[0]

signal_bar_num = 3
intended_fill_bar_num = 4
actual_fill_bar_num = 5
intended_fill_idx = intended_fill_bar_num - 1
actual_fill_idx = actual_fill_bar_num - 1

assert zv_buy_fill["side"] == "buy"
assert toy_zv.loc[intended_fill_idx, "volume"] == 0.0
assert zv_buy_fill["executed_dt"] == toy_zv.loc[actual_fill_idx, "open_time_utc"].tz_localize(None)
assert zv_buy_fill["price"] == toy_zv.loc[actual_fill_idx, "open"]
assert zv_buy_fill["comm"] == expected_commission(zv_buy_fill["size"], zv_buy_fill["price"])

print(f"Signal at bar {signal_bar_num}: dt={toy_zv.loc[signal_bar_num - 1, 'open_time_utc']}, close={toy_zv.loc[signal_bar_num - 1, 'close']}")
print(f"Intended fill at bar {intended_fill_bar_num} open={toy_zv.loc[intended_fill_idx, 'open']}, volume={toy_zv.loc[intended_fill_idx, 'volume']}")
print(f"Actual fill deferred to bar {actual_fill_bar_num} open={toy_zv.loc[actual_fill_idx, 'open']}, volume={toy_zv.loc[actual_fill_idx, 'volume']}")
print(f"Deferred buy actual price: {zv_buy_fill['price']}; expected: {toy_zv.loc[actual_fill_idx, 'open']}")
print("PASS: Deliverable 2 zero-volume deferral validated.")


,open_time_utc,open,close,volume
0,2024-01-01 00:00:00+00:00,100.0,100.0,10.0
1,2024-01-01 01:00:00+00:00,101.0,101.0,10.0
2,2024-01-01 02:00:00+00:00,102.0,102.0,10.0
3,2024-01-01 03:00:00+00:00,103.0,103.0,0.0
4,2024-01-01 04:00:00+00:00,104.0,104.0,10.0
5,2024-01-01 05:00:00+00:00,105.0,105.0,10.0
6,2024-01-01 06:00:00+00:00,106.0,106.0,10.0
7,2024-01-01 07:00:00+00:00,107.0,107.0,10.0


,side,bar_num,dt,close
0,buy,3,2024-01-01 02:00:00,102.0
1,sell,5,2024-01-01 04:00:00,104.0


,status,side,executed_dt,price,size,value,comm
0,Completed,buy,2024-01-01 04:00:00,104.0,1.0,104.0,0.0728
1,Completed,sell,2024-01-01 05:00:00,105.0,-1.0,104.0,0.0735


Signal at bar 3: dt=2024-01-01 02:00:00+00:00, close=102.0
Intended fill at bar 4 open=103.0, volume=0.0
Actual fill deferred to bar 5 open=104.0, volume=10.0
Deferred buy actual price: 104.0; expected: 104.0
PASS: Deliverable 2 zero-volume deferral validated.


### B4. More than 24 zero-volume bars cancels the order

In [9]:
class BuyAndHoldOnBar(bt.Strategy):
    params = (("buy_bar", 3),)

    def __init__(self):
        self.signals = []
        self.fills = []
        self.cancelled = []
        self.order_events = []
        self.final_defer_counts = None
        self.final_pending_live_refs = None
        self.final_submitted_live_refs = None

    def next(self):
        bar_num = len(self)
        dt = self.data.datetime.datetime(0)
        if bar_num == self.p.buy_bar and not self.position:
            order = self.buy()
            self.signals.append({
                "event": "signal_submitted",
                "side": "buy",
                "bar_num": bar_num,
                "dt": dt,
                "close": float(self.data.close[0]),
                "order_ref": order.ref,
            })

    def notify_order(self, order):
        dt = self.data.datetime.datetime(0) if len(self) else None
        event = {
            "event": "notify_order",
            "status": order.getstatusname(),
            "status_code": int(order.status),
            "side": "buy" if order.isbuy() else "sell",
            "ref": order.ref,
            "bar_num_seen_by_strategy": len(self),
            "dt_seen_by_strategy": dt,
            "alive_after_notify": bool(order.alive()),
        }
        if order.status == order.Completed:
            event.update({
                "executed_dt": bt.num2date(order.executed.dt).replace(tzinfo=None),
                "price": float(order.executed.price),
                "size": float(order.executed.size),
                "comm": float(order.executed.comm),
            })
            self.fills.append(event.copy())
        elif order.status == order.Canceled:
            self.cancelled.append(event.copy())
        self.order_events.append(event)

    def stop(self):
        broker = self.broker
        self.final_defer_counts = dict(getattr(broker, "_defer_counts", {}))
        self.final_pending_live_refs = [
            order.ref
            for order in list(getattr(broker, "pending", []))
            if hasattr(order, "alive") and order.alive()
        ]
        self.final_submitted_live_refs = [
            order.ref
            for order in list(getattr(broker, "submitted", []))
            if hasattr(order, "alive") and order.alive()
        ]

long_volumes = [10.0] * 40
for i in range(3, 28):  # 25 zero-volume bars after the buy signal, exceeding MAX_DEFER_BARS=24.
    long_volumes[i] = 0.0

toy_cancel = make_toy_df(n_bars=40, volumes=long_volumes)
toy_cancel_path = write_toy_parquet(toy_cancel, "toy_zero_volume_cancel.parquet")

cancel_strat, _ = run_execution_check(toy_cancel_path, strategy=BuyAndHoldOnBar, buy_bar=3)

signals_cancel_df = pd.DataFrame(cancel_strat.signals)
events_cancel_df = pd.DataFrame(cancel_strat.order_events)
fills_cancel_df = pd.DataFrame(cancel_strat.fills)
cancelled_df = pd.DataFrame(cancel_strat.cancelled)
zero_window = toy_cancel.loc[3:27, ["open_time_utc", "open", "volume"]].copy()
zero_window["bar_num"] = zero_window.index + 1

display(zero_window.head(3))
display(zero_window.tail(3))
display(signals_cancel_df)
display(events_cancel_df)
display(fills_cancel_df)
display(cancelled_df)
print("Final broker defer counts:", cancel_strat.final_defer_counts)
print("Final live pending refs:", cancel_strat.final_pending_live_refs)
print("Final live submitted refs:", cancel_strat.final_submitted_live_refs)

assert len(signals_cancel_df) == 1
assert len(zero_window) == 25
assert (zero_window["volume"] == 0).all()
assert len(cancel_strat.fills) == 0, f"Order should cancel instead of filling, got fills: {cancel_strat.fills}"
assert len(cancel_strat.cancelled) == 1, f"Expected one cancellation, got: {cancel_strat.cancelled}"
assert cancel_strat.cancelled[0]["status"] == "Canceled"
assert cancel_strat.cancelled[0]["alive_after_notify"] is False

statuses = events_cancel_df["status"].tolist()
assert "Submitted" in statuses, statuses
assert "Accepted" in statuses, statuses
assert "Canceled" in statuses, statuses
assert statuses.index("Submitted") < statuses.index("Accepted") < statuses.index("Canceled"), statuses
assert statuses[-1] == "Canceled", statuses

assert cancel_strat.final_defer_counts == {}, cancel_strat.final_defer_counts
assert cancel_strat.final_pending_live_refs == [], cancel_strat.final_pending_live_refs
assert cancel_strat.final_submitted_live_refs == [], cancel_strat.final_submitted_live_refs

print("PASS: Deliverable 2 cancel path validated: 25 zero-volume bars -> Canceled, notify order is sane, no ghost order remains.")


Order 5 cancelled: deferred 25 bars on zero-volume


,open_time_utc,open,volume,bar_num
3,2024-01-01 03:00:00+00:00,103.0,0.0,4
4,2024-01-01 04:00:00+00:00,104.0,0.0,5
5,2024-01-01 05:00:00+00:00,105.0,0.0,6


,open_time_utc,open,volume,bar_num
25,2024-01-02 01:00:00+00:00,125.0,0.0,26
26,2024-01-02 02:00:00+00:00,126.0,0.0,27
27,2024-01-02 03:00:00+00:00,127.0,0.0,28


,event,side,bar_num,dt,close,order_ref
0,signal_submitted,buy,3,2024-01-01 02:00:00,102.0,5


,event,status,status_code,side,ref,bar_num_seen_by_strategy,dt_seen_by_strategy,alive_after_notify
0,notify_order,Submitted,1,buy,5,4,2024-01-01 03:00:00,True
1,notify_order,Accepted,2,buy,5,4,2024-01-01 03:00:00,True
2,notify_order,Canceled,5,buy,5,28,2024-01-02 03:00:00,False


""


,event,status,status_code,side,ref,bar_num_seen_by_strategy,dt_seen_by_strategy,alive_after_notify
0,notify_order,Canceled,5,buy,5,28,2024-01-02 03:00:00,False


Final broker defer counts: {}
Final live pending refs: []
Final live submitted refs: []
PASS: Deliverable 2 cancel path validated: 25 zero-volume bars -> Canceled, notify order is sane, no ghost order remains.


## D1-2 Acceptance Summary

In [10]:
print("Deliverable 1 PASS: ParquetFeed row count, OHLCV values, extra lines, timestamps, and date slicing match pandas.")
print("Deliverable 2 PASS: next-bar-open fills, 7bps commission, zero-volume deferral, and zero-volume cancellation are validated.")
print("Reminder: keep pytest as the primary correctness evidence; this notebook is the readable acceptance spot-check.")
print("Scope note: Deliverables 1/2 are accepted pending Phase 1A manual trade audit on real engine-generated trades after Deliverable 3.")


Deliverable 1 PASS: ParquetFeed row count, OHLCV values, extra lines, timestamps, and date slicing match pandas.
Deliverable 2 PASS: next-bar-open fills, 7bps commission, zero-volume deferral, and zero-volume cancellation are validated.
Reminder: keep pytest as the primary correctness evidence; this notebook is the readable acceptance spot-check.
Scope note: Deliverables 1/2 are accepted pending Phase 1A manual trade audit on real engine-generated trades after Deliverable 3.


## Part C - Deliverables 3-8 Acceptance

This section is the acceptance harness for Deliverables 3-8. It intentionally runs a real 2024 SMA crossover backtest and verifies four layers:

1. Automated tests are green before acceptance.
2. One real single-run result is sane.
3. Three real trades are manually audited against raw parquet bars.
4. Registry, trade artifact, metrics, and in-memory result agree.

Running this section creates ignored runtime artifacts: `backtest/experiments.db` and `data/results/trades_{run_id}.csv`. That is expected and required for acceptance.

### C1. Automated test gate

Run the Phase 1A-specific tests before trusting the notebook acceptance checks.

In [11]:
import subprocess
import sys
from datetime import datetime, timezone
import json as jsonlib
import math

D38_TEST_COMMAND = [
    sys.executable,
    "-m",
    "pytest",
    "tests/test_engine.py",
    "tests/test_metrics.py",
    "tests/test_experiment_registry.py",
    "tests/test_execution_model.py",
    "tests/test_bt_parquet_feed.py",
]

completed = subprocess.run(D38_TEST_COMMAND, capture_output=True, text=True)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
assert completed.returncode == 0, "D3-8 automated test gate failed"

D38_ACCEPTANCE = {"automated_tests": True}
print("PASS: Phase 1A automated test gate passed.")


============================= test session starts ==============================
platform darwin -- Python 3.11.8, pytest-9.0.2, pluggy-1.6.0
rootdir: /Users/yutianyang/Documents/GitHub/btc-alpha-pipeline
configfile: pyproject.toml
plugins: anyio-4.13.0
collected 90 items

tests/test_engine.py ................                                    [ 17%]
tests/test_metrics.py .....................                              [ 41%]
tests/test_experiment_registry.py ..........................             [ 70%]
tests/test_execution_model.py .................                          [ 88%]
tests/test_bt_parquet_feed.py ..........                                 [100%]

============================== 90 passed in 1.43s ==============================

PASS: Phase 1A automated test gate passed.


### C2. Run one real 2024 SMA crossover backtest

This is the core D3/D6/D7 acceptance run. It uses the real canonical parquet, real `ParquetFeed`, real `configure_cerebro`, real `SMACrossover`, real metrics, real registry write, and real trade CSV artifact.

In [12]:
from pathlib import Path
from datetime import datetime, timezone
import math

import pandas as pd
from IPython.display import display

PARQUET_PATH = globals().get("PARQUET_PATH", Path("data/raw/btcusdt_1h.parquet"))
COMMISSION_RATE = globals().get("COMMISSION_RATE", 0.0007)
assert PARQUET_PATH.exists(), f"Missing canonical parquet: {PARQUET_PATH}"

import importlib

import backtest.engine as engine_module
import backtest.experiment_registry as registry_module
import backtest.trade_audit as trade_audit_module
import strategies.baseline.sma_crossover as sma_module

# Reload these modules so a notebook kernel opened before `git pull` does not
# keep stale Phase 1A code in memory.
engine_module = importlib.reload(engine_module)
registry_module = importlib.reload(registry_module)
trade_audit_module = importlib.reload(trade_audit_module)
sma_module = importlib.reload(sma_module)

run_backtest = engine_module.run_backtest
DEFAULT_DB_PATH = registry_module.DEFAULT_DB_PATH
get_connection = registry_module.get_connection
get_run = registry_module.get_run
audit_trade = trade_audit_module.audit_trade
load_ohlcv = trade_audit_module.load_ohlcv
SMACrossover = sma_module.SMACrossover

D38_START = datetime(2024, 1, 1, tzinfo=timezone.utc)
# Match CLI date-only semantics: --end 2024-12-31 includes all hourly bars on 2024-12-31.
D38_END = datetime(2024, 12, 31, 23, tzinfo=timezone.utc)
D38_STRATEGY_PARAMS = {"fast_period": 20, "slow_period": 50}

print("Using engine module:", engine_module.__file__)
print("Using SMA module:", sma_module.__file__)
print("Using parquet:", PARQUET_PATH.resolve())
print("Acceptance range:", D38_START, "to", D38_END)

d38_raw_df = pd.read_parquet(PARQUET_PATH).sort_values("open_time_utc").reset_index(drop=True)
d38_start_ts = pd.Timestamp(D38_START)
d38_end_ts = pd.Timestamp(D38_END)
d38_raw_range = d38_raw_df[
    (d38_raw_df["open_time_utc"] >= d38_start_ts) &
    (d38_raw_df["open_time_utc"] <= d38_end_ts)
].reset_index(drop=True)
D38_EXPECTED_RAW_BARS = len(d38_raw_range)
D38_EXPECTED_FIRST_BAR = d38_raw_range["open_time_utc"].iloc[0].tz_localize(None)
D38_EXPECTED_LAST_BAR = d38_raw_range["open_time_utc"].iloc[-1].tz_localize(None)
assert D38_EXPECTED_RAW_BARS == 8784, f"Expected full 2024 hourly range to have 8784 bars, got {D38_EXPECTED_RAW_BARS}"
assert D38_EXPECTED_FIRST_BAR == pd.Timestamp("2024-01-01 00:00:00")
assert D38_EXPECTED_LAST_BAR == pd.Timestamp("2024-12-31 23:00:00")

result = run_backtest(
    strategy_cls=SMACrossover,
    start_date=D38_START,
    end_date=D38_END,
    strategy_params=D38_STRATEGY_PARAMS,
    parquet_path=PARQUET_PATH,
    cash=10_000.0,
    write_registry=True,
)

D38_RUN_ID = result.run_id
D38_TRADE_CSV = Path(result.trade_csv_path) if result.trade_csv_path else None

summary_rows = [
    {"field": "run_id", "value": result.run_id},
    {"field": "strategy_name", "value": result.strategy_name},
    {"field": "start_date", "value": result.start_date},
    {"field": "end_date", "value": result.end_date},
    {"field": "warmup_bars", "value": result.warmup_bars},
    {"field": "effective_start", "value": result.effective_start},
    {"field": "trade_csv_path", "value": str(D38_TRADE_CSV)},
    {"field": "expected_raw_bars", "value": D38_EXPECTED_RAW_BARS},
    {"field": "expected_first_bar", "value": D38_EXPECTED_FIRST_BAR},
    {"field": "expected_last_bar", "value": D38_EXPECTED_LAST_BAR},
    {"field": "in_memory_trade_count", "value": len(result.trades)},
    {"field": "metrics_total_trades", "value": result.metrics["total_trades"]},
    {"field": "equity_curve_length", "value": len(result.equity_curve)},
    {"field": "total_return", "value": result.metrics["total_return"]},
    {"field": "sharpe_ratio", "value": result.metrics["sharpe_ratio"]},
    {"field": "max_drawdown", "value": result.metrics["max_drawdown"]},
    {"field": "win_rate", "value": result.metrics["win_rate"]},
    {"field": "profit_factor", "value": result.metrics["profit_factor"]},
]
display(pd.DataFrame(summary_rows))

if D38_TRADE_CSV is None or not D38_TRADE_CSV.exists():
    diagnostic = {
        "run_id": result.run_id,
        "strategy_name": result.strategy_name,
        "trades_len": len(result.trades),
        "metrics_total_trades": result.metrics.get("total_trades"),
        "trade_csv_path": str(D38_TRADE_CSV),
        "equity_curve_length": len(result.equity_curve),
        "effective_start": result.effective_start,
        "hint": "Restart the notebook kernel and run from the top after pulling latest code. A fresh process should produce a non-empty SMA trade CSV for 2024.",
    }
    display(pd.DataFrame([diagnostic]).T.rename(columns={0: "value"}))
    raise AssertionError("Trade CSV artifact missing because this run produced no completed trades or no CSV path. See diagnostic table above.")

assert result.strategy_name == "sma_crossover"
assert result.warmup_bars == 50
assert result.effective_start is not None
assert len(result.equity_curve) > 0
assert result.start_date == D38_START
assert result.end_date == D38_END
assert result.equity_curve.index[-1] == D38_EXPECTED_LAST_BAR
assert len(result.equity_curve) == D38_EXPECTED_RAW_BARS - result.warmup_bars
assert result.metrics["total_trades"] > 0, "SMA crossover produced no completed round-trip trades"
assert len(result.trades) == result.metrics["total_trades"]
assert 5 <= result.metrics["total_trades"] <= 500, "SMA trade count outside sane annual range"
assert -1.0 < result.metrics["total_return"] < 1.0, "Total return outside broad sanity range"
assert math.isfinite(result.metrics["sharpe_ratio"])
assert result.metrics["sharpe_ratio"] < 2.0, "Suspiciously high Sharpe for baseline SMA"

D38_ACCEPTANCE.update({"D3_real_run": True, "D6_template_consumed": True, "D7_sma_sane": True})
print("PASS: Real SMA single-run completed with sane baseline outputs.")


2026-04-16T04:21:02Z [INFO] Starting backtest: strategy=sma_crossover, range=[2024-01-01, 2024-12-31], run_id=5ef19628
2026-04-16T04:21:02Z [INFO] Loading parquet feed: data/raw/btcusdt_1h.parquet
2026-04-16T04:21:02Z [INFO]   Loaded 8784 bars: 2024-01-01 00:00:00 to 2024-12-31 23:00:00
2026-04-16T04:21:02Z [INFO] Cost model: effective_7bps_per_side (fee=4.0bps + slip=3.0bps = 7.0bps per side)
2026-04-16T04:21:02Z [INFO] Cerebro configured: cash=10000.00, coc=False, coo=False, effective_7bps_per_side, sizer=99%


Using engine module: /Users/yutianyang/Documents/GitHub/btc-alpha-pipeline/backtest/engine.py
Using SMA module: /Users/yutianyang/Documents/GitHub/btc-alpha-pipeline/strategies/baseline/sma_crossover.py
Using parquet: /Users/yutianyang/Documents/GitHub/btc-alpha-pipeline/data/raw/btcusdt_1h.parquet
Acceptance range: 2024-01-01 00:00:00+00:00 to 2024-12-31 23:00:00+00:00


2026-04-16T04:21:04Z [INFO] Metrics: return=0.3483 sharpe=1.027 maxDD=0.3239 trades=105 win_rate=0.43 PF=1.26
2026-04-16T04:21:04Z [INFO] Trade log saved: /Users/yutianyang/Documents/GitHub/btc-alpha-pipeline/data/results/trades_5ef19628-3a4d-47fd-9cd2-e390df6bd668.csv (105 trades)
2026-04-16T04:21:04Z [INFO] Ensured 'runs' table exists with Phase 1A schema
2026-04-16T04:21:04Z [INFO] Inserted run 5ef19628-3a4d-47fd-9cd2-e390df6bd668 (strategy: sma_crossover)
2026-04-16T04:21:04Z [INFO] Backtest complete: 105 trades, return=0.3483, sharpe=1.027, csv=/Users/yutianyang/Documents/GitHub/btc-alpha-pipeline/data/results/trades_5ef19628-3a4d-47fd-9cd2-e390df6bd668.csv


,field,value
0,run_id,5ef19628-3a4d-47fd-9cd2-e390df6bd668
1,strategy_name,sma_crossover
2,start_date,2024-01-01 00:00:00+00:00
3,end_date,2024-12-31 23:00:00+00:00
4,warmup_bars,50
5,effective_start,2024-01-03 02:00:00
6,trade_csv_path,/Users/yutianyang/Documents/GitHub/btc-alpha-p...
7,expected_raw_bars,8784
8,expected_first_bar,2024-01-01 00:00:00
9,expected_last_bar,2024-12-31 23:00:00


PASS: Real SMA single-run completed with sane baseline outputs.


### C3. Registry row and trade CSV consistency

This validates D3 artifact creation and D5 registry semantics for Phase 1A single-run mode.

In [13]:
from pathlib import Path
import json as jsonlib

import pandas as pd

assert "D38_RUN_ID" in globals(), "Run C2 first to create D38_RUN_ID."
assert "D38_TRADE_CSV" in globals(), "Run C2 first to create D38_TRADE_CSV."
assert D38_TRADE_CSV is not None and Path(D38_TRADE_CSV).exists(), (
    "Trade CSV is missing. Re-run C2 from a fresh kernel; C2 now prints diagnostics if the SMA run produces no CSV."
)

conn = get_connection(DEFAULT_DB_PATH)
try:
    registry_row = get_run(conn, D38_RUN_ID)
finally:
    conn.close()

assert registry_row is not None, f"Registry row missing for run_id={D38_RUN_ID}"
trades_df = pd.read_csv(D38_TRADE_CSV)

required_trade_cols = [
    "trade_id",
    "entry_signal_time_utc",
    "entry_time_utc",
    "entry_price",
    "entry_bar_volume",
    "exit_signal_time_utc",
    "exit_time_utc",
    "exit_price",
    "exit_bar_volume",
    "size",
    "entry_commission",
    "exit_commission",
    "total_commission",
    "pnl",
    "pnl_pct",
]
missing_trade_cols = sorted(set(required_trade_cols) - set(trades_df.columns))
assert not missing_trade_cols, f"Trade CSV missing columns: {missing_trade_cols}"

phase1a_null_fields = ["train_start", "train_end", "validation_start", "validation_end"]
registry_checks = []

def add_registry_check(name, expected, actual, passed=None):
    if passed is None:
        passed = actual == expected
    registry_checks.append({"check": name, "expected": expected, "actual": actual, "pass": bool(passed)})

add_registry_check("run_id", D38_RUN_ID, registry_row["run_id"])
add_registry_check("run_type", "single_run", registry_row["run_type"])
add_registry_check("strategy_name", "sma_crossover", registry_row["strategy_name"])
add_registry_check("strategy_source", "manual", registry_row["strategy_source"])
expected_notes = jsonlib.dumps(D38_STRATEGY_PARAMS)
add_registry_check("notes contains strategy params", expected_notes, registry_row["notes"])
add_registry_check("fee_model", "effective_7bps_per_side", registry_row["fee_model"])
add_registry_check("warmup_bars", result.warmup_bars, registry_row["warmup_bars"])
add_registry_check("test_start", "2024-01-01T00:00:00Z", registry_row["test_start"])
add_registry_check("test_end", "2024-12-31T23:00:00Z", registry_row["test_end"])
add_registry_check("effective_start", result.effective_start.strftime("%Y-%m-%dT%H:%M:%SZ"), registry_row["effective_start"])
for field in phase1a_null_fields:
    add_registry_check(field, None, registry_row[field], registry_row[field] is None)
add_registry_check("total_trades == trade CSV rows", len(trades_df), registry_row["total_trades"])
add_registry_check("total_trades == result metrics", result.metrics["total_trades"], registry_row["total_trades"])

registry_df = pd.DataFrame(registry_checks)
display(registry_df)
display(trades_df.head())
print("Trade CSV path:", D38_TRADE_CSV)
print("Trade CSV rows:", len(trades_df))

assert registry_df["pass"].all(), registry_df[~registry_df["pass"]]
assert len(trades_df) == result.metrics["total_trades"] == registry_row["total_trades"]

D38_ACCEPTANCE.update({"D3_artifacts": True, "D5_registry": True})
print("PASS: Registry row, trade CSV, and result metrics agree.")


,check,expected,actual,pass
0,run_id,5ef19628-3a4d-47fd-9cd2-e390df6bd668,5ef19628-3a4d-47fd-9cd2-e390df6bd668,True
1,run_type,single_run,single_run,True
2,strategy_name,sma_crossover,sma_crossover,True
3,strategy_source,manual,manual,True
4,notes contains strategy params,"{""fast_period"": 20, ""slow_period"": 50}","{""fast_period"": 20, ""slow_period"": 50}",True
5,fee_model,effective_7bps_per_side,effective_7bps_per_side,True
6,warmup_bars,50,50,True
7,test_start,2024-01-01T00:00:00Z,2024-01-01T00:00:00Z,True
8,test_end,2024-12-31T23:00:00Z,2024-12-31T23:00:00Z,True
9,effective_start,2024-01-03T02:00:00Z,2024-01-03T02:00:00Z,True


,trade_id,entry_signal_time_utc,entry_time_utc,entry_price,entry_bar_volume,exit_signal_time_utc,exit_time_utc,exit_price,exit_bar_volume,size,entry_commission,exit_commission,total_commission,pnl,pnl_pct
0,1,2024-01-05T02:00:00Z,2024-01-05T03:00:00Z,43506.01,1543.67157,2024-01-06T10:00:00Z,2024-01-06T11:00:00Z,43694.47,396.12062,0.227555,6.930002,6.960021,13.890023,28.994959,0.002929
1,2,2024-01-06T12:00:00Z,2024-01-06T13:00:00Z,43607.95,473.09350,2024-01-06T13:00:00Z,2024-01-06T14:00:00Z,43704.08,998.40603,0.227681,6.950092,6.965413,13.915505,7.971472,0.000803
2,3,2024-01-07T00:00:00Z,2024-01-07T01:00:00Z,44119.11,668.13600,2024-01-08T09:00:00Z,2024-01-08T10:00:00Z,43720.45,1256.15965,0.225222,6.955618,6.892767,13.848385,-103.635410,-0.010430
3,4,2024-01-08T12:00:00Z,2024-01-08T13:00:00Z,45111.10,3323.31376,2024-01-10T10:00:00Z,2024-01-10T11:00:00Z,45532.01,2363.70366,0.217995,6.883800,6.948029,13.831829,77.924499,0.007924
4,5,2024-01-11T09:00:00Z,2024-01-11T10:00:00Z,46220.12,2559.25355,2024-01-12T10:00:00Z,2024-01-12T11:00:00Z,46063.62,1976.90053,0.214434,6.937800,6.914309,13.852109,-47.410955,-0.004784


Trade CSV path: /Users/yutianyang/Documents/GitHub/btc-alpha-pipeline/data/results/trades_5ef19628-3a4d-47fd-9cd2-e390df6bd668.csv
Trade CSV rows: 105
PASS: Registry row, trade CSV, and result metrics agree.


### C4. Manual audit of first, middle, and last real trades

This is the D8 acceptance check. It audits three real trades against raw parquet bars and verifies next-bar-open fill semantics, nonzero fill volume, commissions, total commission, and PnL math.

In [14]:
import numpy as np
import pandas as pd

COMMISSION_RATE = globals().get("COMMISSION_RATE", 0.0007)

ohlcv = load_ohlcv(PARQUET_PATH)
trade_indices = sorted(set([0, len(trades_df) // 2, len(trades_df) - 1]))
audit_rows = []

for idx in trade_indices:
    trade = trades_df.iloc[idx]
    audit = audit_trade(trade, ohlcv, context_bars=2)
    module_checks_pass = all(check.get("pass", False) for check in audit["checks"].values())

    entry_signal_dt = pd.Timestamp(trade["entry_signal_time_utc"]).tz_localize(None)
    entry_fill_dt = pd.Timestamp(trade["entry_time_utc"]).tz_localize(None)
    exit_signal_dt = pd.Timestamp(trade["exit_signal_time_utc"]).tz_localize(None)
    exit_fill_dt = pd.Timestamp(trade["exit_time_utc"]).tz_localize(None)

    entry_signal_idx = ohlcv.index.get_loc(entry_signal_dt)
    entry_fill_idx = ohlcv.index.get_loc(entry_fill_dt)
    exit_signal_idx = ohlcv.index.get_loc(exit_signal_dt)
    exit_fill_idx = ohlcv.index.get_loc(exit_fill_dt)

    expected_entry_open = float(ohlcv.loc[entry_fill_dt, "open"])
    expected_exit_open = float(ohlcv.loc[exit_fill_dt, "open"])
    expected_entry_comm = float(abs(trade["size"]) * trade["entry_price"] * COMMISSION_RATE)
    expected_exit_comm = float(abs(trade["size"]) * trade["exit_price"] * COMMISSION_RATE)
    expected_total_comm = expected_entry_comm + expected_exit_comm
    expected_pnl = float((trade["exit_price"] - trade["entry_price"]) * abs(trade["size"]) - expected_total_comm)
    expected_pnl_pct = expected_pnl / float(abs(trade["size"]) * trade["entry_price"])

    checks = {
        "module_audit_checks_pass": module_checks_pass,
        "entry_signal_before_fill": entry_signal_dt < entry_fill_dt,
        "entry_next_bar": entry_fill_idx - entry_signal_idx == 1,
        "entry_price_open_match": np.isclose(trade["entry_price"], expected_entry_open, atol=1e-8),
        "entry_volume_nonzero": float(trade["entry_bar_volume"]) > 0,
        "exit_signal_before_fill": exit_signal_dt < exit_fill_dt,
        "exit_next_bar": exit_fill_idx - exit_signal_idx == 1,
        "exit_price_open_match": np.isclose(trade["exit_price"], expected_exit_open, atol=1e-8),
        "exit_volume_nonzero": float(trade["exit_bar_volume"]) > 0,
        "total_commission_math": np.isclose(trade["entry_commission"] + trade["exit_commission"], trade["total_commission"], atol=1e-8),
        "entry_commission_7bps": np.isclose(trade["entry_commission"], expected_entry_comm, atol=1e-8),
        "exit_commission_7bps": np.isclose(trade["exit_commission"], expected_exit_comm, atol=1e-8),
        "pnl_math": np.isclose(trade["pnl"], expected_pnl, atol=1e-6),
        "pnl_pct_math": np.isclose(trade["pnl_pct"], expected_pnl_pct, atol=1e-10),
    }

    for check_name, passed in checks.items():
        audit_rows.append({
            "trade_csv_index": idx,
            "trade_id": int(trade["trade_id"]),
            "check": check_name,
            "pass": bool(passed),
        })

    print(f"Trade #{int(trade['trade_id'])}: entry {entry_signal_dt} -> {entry_fill_dt} @ {trade['entry_price']}; exit {exit_signal_dt} -> {exit_fill_dt} @ {trade['exit_price']}")
    print(f"  expected entry open={expected_entry_open}, expected exit open={expected_exit_open}")
    print(f"  expected total commission={expected_total_comm:.8f}, actual={trade['total_commission']:.8f}")
    print(f"  expected pnl={expected_pnl:.8f}, actual={trade['pnl']:.8f}")

audit_df = pd.DataFrame(audit_rows)
display(audit_df)
assert audit_df["pass"].all(), audit_df[~audit_df["pass"]]

D38_ACCEPTANCE["D8_trade_audit"] = True
print("PASS: First, middle, and last real trades audit cleanly against raw parquet.")


Trade #1: entry 2024-01-05 02:00:00 -> 2024-01-05 03:00:00 @ 43506.01; exit 2024-01-06 10:00:00 -> 2024-01-06 11:00:00 @ 43694.47
  expected entry open=43506.01, expected exit open=43694.47
  expected total commission=13.89002267, actual=13.89002267
  expected pnl=28.99495871, actual=28.99495871
Trade #53: entry 2024-06-20 09:00:00 -> 2024-06-20 10:00:00 @ 65719.99; exit 2024-06-21 03:00:00 -> 2024-06-21 04:00:00 @ 64506.15
  expected entry open=65719.99, expected exit open=64506.15
  expected total commission=14.20644646, actual=14.20644646
  expected pnl=-203.37576352, actual=-203.37576352
Trade #105: entry 2024-12-29 08:00:00 -> 2024-12-29 09:00:00 @ 95123.33; exit 2024-12-29 21:00:00 -> 2024-12-29 22:00:00 @ 93352.65
  expected entry open=95123.33, expected exit open=93352.65
  expected total commission=19.00943919, actual=19.00943919
  expected pnl=-274.13580133, actual=-274.13580133


,trade_csv_index,trade_id,check,pass
0,0,1,module_audit_checks_pass,True
1,0,1,entry_signal_before_fill,True
2,0,1,entry_next_bar,True
3,0,1,entry_price_open_match,True
4,0,1,entry_volume_nonzero,True
5,0,1,exit_signal_before_fill,True
6,0,1,exit_next_bar,True
7,0,1,exit_price_open_match,True
8,0,1,exit_volume_nonzero,True
9,0,1,total_commission_math,True


PASS: First, middle, and last real trades audit cleanly against raw parquet.


### C5. Warmup boundary validation

This checks that warmup does not pollute metrics: `effective_start` should be the first post-warmup bar, no trade signal/fill should occur before it, and the equity curve length should equal the post-warmup bar count.

In [15]:
import pandas as pd

acceptance_raw = (
    pd.read_parquet(PARQUET_PATH)
    .sort_values("open_time_utc")
    .reset_index(drop=True)
)
run_raw = acceptance_raw[
    (acceptance_raw["open_time_utc"] >= pd.Timestamp(D38_START)) &
    (acceptance_raw["open_time_utc"] <= pd.Timestamp(D38_END))
].reset_index(drop=True)

# Backtrader records the first equity point after the warmup rows are complete.
# With WARMUP_BARS=50 and zero-based pandas indexing, that is iloc[50].
expected_effective_start = run_raw["open_time_utc"].iloc[result.warmup_bars].tz_localize(None)
actual_effective_start = pd.Timestamp(result.effective_start).tz_localize(None)
post_warmup_bar_count = len(run_raw) - result.warmup_bars
first_entry_signal = pd.Timestamp(trades_df["entry_signal_time_utc"].min()).tz_localize(None)
first_entry_fill = pd.Timestamp(trades_df["entry_time_utc"].min()).tz_localize(None)

equity_curve = result.equity_curve.copy()
warmup_summary = pd.DataFrame([
    {"check": "raw bars in run range", "value": len(run_raw)},
    {"check": "warmup bars", "value": result.warmup_bars},
    {"check": "expected effective_start", "value": expected_effective_start},
    {"check": "actual effective_start", "value": actual_effective_start},
    {"check": "post-warmup bar count", "value": post_warmup_bar_count},
    {"check": "equity curve length", "value": len(equity_curve)},
    {"check": "first trade signal", "value": first_entry_signal},
    {"check": "first trade fill", "value": first_entry_fill},
])
display(warmup_summary)

assert actual_effective_start == expected_effective_start
assert len(equity_curve) == post_warmup_bar_count
assert first_entry_signal >= actual_effective_start
assert first_entry_fill > actual_effective_start
assert equity_curve.index.min() == expected_effective_start
assert equity_curve.index.max() == run_raw["open_time_utc"].iloc[-1].tz_localize(None)

D38_ACCEPTANCE["D3_warmup"] = True
D38_ACCEPTANCE["D4_no_warmup_pollution"] = True
print("PASS: Warmup boundary, first trade timing, and equity curve length are consistent.")


,check,value
0,raw bars in run range,8784
1,warmup bars,50
2,expected effective_start,2024-01-03 02:00:00
3,actual effective_start,2024-01-03 02:00:00
4,post-warmup bar count,8734
5,equity curve length,8734
6,first trade signal,2024-01-05 02:00:00
7,first trade fill,2024-01-05 03:00:00


PASS: Warmup boundary, first trade timing, and equity curve length are consistent.


### C6. Metrics cross-checks

This checks D4/D5 self-consistency without reimplementing every metric: capital return, trade counts, win rate, average trade return, profit factor, and DB metric values must agree with the result object and trade CSV.

In [16]:
import math

import numpy as np
import pandas as pd

metric_checks = []

def add_metric_check(name, expected, actual, passed=None):
    if passed is None:
        if isinstance(expected, (float, np.floating)) or isinstance(actual, (float, np.floating)):
            passed = np.isclose(float(actual), float(expected), rtol=1e-8, atol=1e-8)
        else:
            passed = actual == expected
    metric_checks.append({"check": name, "expected": expected, "actual": actual, "pass": bool(passed)})

initial_capital = float(result.metrics["initial_capital"])
final_capital = float(result.metrics["final_capital"])
total_return = float(result.metrics["total_return"])
expected_final_capital = initial_capital * (1.0 + total_return)

pnls = trades_df["pnl"].astype(float)
pnl_pcts = trades_df["pnl_pct"].astype(float)
gross_profit = float(pnls[pnls > 0].sum())
gross_loss = float(abs(pnls[pnls < 0].sum()))
expected_profit_factor = gross_profit / gross_loss if gross_loss > 0 else float("inf")
expected_win_rate = float((pnls > 0).sum() / len(pnls))
expected_avg_trade_return = float(pnl_pcts.mean())

add_metric_check("final_capital = initial_capital * (1 + total_return)", expected_final_capital, final_capital)
add_metric_check("total_trades equals trade CSV rows", len(trades_df), result.metrics["total_trades"])
add_metric_check("win_rate equals winning trades / total trades", expected_win_rate, result.metrics["win_rate"])
add_metric_check("avg_trade_return equals mean trade pnl_pct", expected_avg_trade_return, result.metrics["avg_trade_return"])
add_metric_check("profit_factor equals gross profits / gross losses", expected_profit_factor, result.metrics["profit_factor"])

for key in [
    "initial_capital",
    "final_capital",
    "total_return",
    "sharpe_ratio",
    "max_drawdown",
    "max_drawdown_duration_hours",
    "total_trades",
    "win_rate",
    "avg_trade_duration_hours",
    "avg_trade_return",
    "profit_factor",
]:
    add_metric_check(f"registry {key} equals result metric", result.metrics[key], registry_row[key])

finite_metric_keys = [
    "initial_capital",
    "final_capital",
    "total_return",
    "sharpe_ratio",
    "max_drawdown",
    "max_drawdown_duration_hours",
    "win_rate",
    "avg_trade_duration_hours",
    "avg_trade_return",
    "profit_factor",
]
for key in finite_metric_keys:
    add_metric_check(f"{key} is finite", True, math.isfinite(float(result.metrics[key])))

metric_df = pd.DataFrame(metric_checks)
display(metric_df)
assert metric_df["pass"].all(), metric_df[~metric_df["pass"]]

D38_ACCEPTANCE["D4_metrics"] = True
D38_ACCEPTANCE["D5_metrics_registry_consistency"] = True
print("PASS: Metrics are finite and self-consistent with trade CSV, capital, and registry values.")


,check,expected,actual,pass
0,final_capital = initial_capital * (1 + total_r...,13482.856871,13482.856871,True
1,total_trades equals trade CSV rows,105,105,True
2,win_rate equals winning trades / total trades,0.428571,0.428571,True
3,avg_trade_return equals mean trade pnl_pct,0.003823,0.003823,True
4,profit_factor equals gross profits / gross losses,1.259227,1.259227,True
5,registry initial_capital equals result metric,10000.0,10000.0,True
6,registry final_capital equals result metric,13482.856871,13482.856871,True
7,registry total_return equals result metric,0.348286,0.348286,True
8,registry sharpe_ratio equals result metric,1.026653,1.026653,True
9,registry max_drawdown equals result metric,0.323926,0.323926,True


PASS: Metrics are finite and self-consistent with trade CSV, capital, and registry values.


### C7. D3-8 final acceptance summary

In [17]:
deliverable_rows = [
    {
        "deliverable": "D3 engine.py",
        "pass": bool(D38_ACCEPTANCE.get("D3_real_run") and D38_ACCEPTANCE.get("D3_artifacts") and D38_ACCEPTANCE.get("D3_warmup")),
        "evidence": "real SMA run completed; registry row and trade CSV exist; warmup/equity timing verified",
    },
    {
        "deliverable": "D4 metrics.py",
        "pass": bool(D38_ACCEPTANCE.get("D4_metrics") and D38_ACCEPTANCE.get("D4_no_warmup_pollution")),
        "evidence": "metrics finite; capital/trade/profit-factor checks agree; equity curve excludes warmup",
    },
    {
        "deliverable": "D5 experiment_registry.py",
        "pass": bool(D38_ACCEPTANCE.get("D5_registry") and D38_ACCEPTANCE.get("D5_metrics_registry_consistency")),
        "evidence": "single_run row present; Phase 1A train/validation NULL; fee/warmup/test fields correct",
    },
    {
        "deliverable": "D6 template.py",
        "pass": bool(D38_ACCEPTANCE.get("D6_template_consumed")),
        "evidence": "engine consumed strategy name, params, and WARMUP_BARS from SMA strategy/template interface",
    },
    {
        "deliverable": "D7 sma_crossover.py",
        "pass": bool(D38_ACCEPTANCE.get("D7_sma_sane")),
        "evidence": "SMA generated a sane number of trades and non-inflated baseline metrics",
    },
    {
        "deliverable": "D8 trade_audit.py",
        "pass": bool(D38_ACCEPTANCE.get("D8_trade_audit")),
        "evidence": "first/middle/last real trades audited against raw parquet with next-bar-open and commission checks",
    },
]

deliverable_df = pd.DataFrame(deliverable_rows)
display(deliverable_df)
assert deliverable_df["pass"].all(), deliverable_df[~deliverable_df["pass"]]

print("D3-8 ACCEPTANCE PASS")
print("Run ID:", D38_RUN_ID)
print("Trade CSV:", D38_TRADE_CSV)
print("Scope note: AlphaBroker multi-trade stress confidence comes from this real SMA run plus later D8/D10-style audits; this notebook keeps the evidence linked to registry, artifacts, and metrics.")


,deliverable,pass,evidence
0,D3 engine.py,True,real SMA run completed; registry row and trade...
1,D4 metrics.py,True,metrics finite; capital/trade/profit-factor ch...
2,D5 experiment_registry.py,True,single_run row present; Phase 1A train/validat...
3,D6 template.py,True,"engine consumed strategy name, params, and WAR..."
4,D7 sma_crossover.py,True,SMA generated a sane number of trades and non-...
5,D8 trade_audit.py,True,first/middle/last real trades audited against ...


D3-8 ACCEPTANCE PASS
Run ID: 5ef19628-3a4d-47fd-9cd2-e390df6bd668
Trade CSV: /Users/yutianyang/Documents/GitHub/btc-alpha-pipeline/data/results/trades_5ef19628-3a4d-47fd-9cd2-e390df6bd668.csv
Scope note: AlphaBroker multi-trade stress confidence comes from this real SMA run plus later D8/D10-style audits; this notebook keeps the evidence linked to registry, artifacts, and metrics.


## Part D - Deliverables 9-10 Lightweight Summary

D9 introduces a second baseline (`momentum`) with different trade timing than SMA crossover: threshold-based, more frequent, and different holding rhythms.

This appendix is intentionally small. D1-D8 already validate the execution and artifact machinery in detail; D10's automated end-to-end test is the primary coverage for the full Phase 1A path. These cells leave a human-readable record that momentum runs sanely and that the D10 test suite passes.

### D9. Momentum baseline one-run summary

Run momentum once over the same full-year 2024 range used in the D3-D8 acceptance run. This checks that the second baseline generates trades, has a reasonable frequency, and produces finite baseline metrics.

In [18]:
from pathlib import Path
from datetime import datetime, timezone
import importlib
import math

import pandas as pd
from IPython.display import display

PARQUET_PATH = globals().get("PARQUET_PATH", Path("data/raw/btcusdt_1h.parquet"))
assert PARQUET_PATH.exists(), f"Missing canonical parquet: {PARQUET_PATH}"

import backtest.engine as engine_module
import strategies.baseline.momentum as momentum_module

engine_module = importlib.reload(engine_module)
momentum_module = importlib.reload(momentum_module)
run_backtest = engine_module.run_backtest
Momentum = momentum_module.Momentum

D9_START = datetime(2024, 1, 1, tzinfo=timezone.utc)
D9_END = datetime(2024, 12, 31, 23, tzinfo=timezone.utc)
D9_PARAMS = {"lookback_period": 24, "entry_threshold": 0.02, "exit_threshold": 0.0}

momentum_result = run_backtest(
    strategy_cls=Momentum,
    start_date=D9_START,
    end_date=D9_END,
    strategy_params=D9_PARAMS,
    parquet_path=PARQUET_PATH,
    cash=10_000.0,
    write_registry=True,
)

D9_TRADE_CSV = Path(momentum_result.trade_csv_path) if momentum_result.trade_csv_path else None

momentum_summary = pd.DataFrame([
    {"field": "strategy_name", "value": momentum_result.strategy_name},
    {"field": "run_id", "value": momentum_result.run_id},
    {"field": "range", "value": f"{D9_START.isoformat()} -> {D9_END.isoformat()}"},
    {"field": "params", "value": D9_PARAMS},
    {"field": "trades", "value": momentum_result.metrics["total_trades"]},
    {"field": "sharpe_ratio", "value": momentum_result.metrics["sharpe_ratio"]},
    {"field": "total_return", "value": momentum_result.metrics["total_return"]},
    {"field": "warmup_bars", "value": momentum_result.warmup_bars},
    {"field": "effective_start", "value": momentum_result.effective_start},
    {"field": "trade_csv", "value": str(D9_TRADE_CSV)},
])
display(momentum_summary)

assert momentum_result.strategy_name == "momentum"
assert momentum_result.warmup_bars == 24
assert momentum_result.effective_start is not None
assert momentum_result.metrics["total_trades"] > 0, "Momentum produced no trades"
assert 20 <= momentum_result.metrics["total_trades"] <= 1000, "Momentum trade frequency outside broad sanity range"
assert math.isfinite(momentum_result.metrics["sharpe_ratio"])
assert math.isfinite(momentum_result.metrics["total_return"])
assert -1.0 < momentum_result.metrics["total_return"] < 2.0, "Momentum return outside broad sanity range"
assert momentum_result.metrics["sharpe_ratio"] < 2.0, "Suspiciously high momentum Sharpe for baseline"
assert D9_TRADE_CSV is not None and D9_TRADE_CSV.exists(), "Momentum trade CSV missing"

print("D9 MOMENTUM SUMMARY PASS")
print(f"Momentum run_id={momentum_result.run_id}, trades={momentum_result.metrics['total_trades']}, Sharpe={momentum_result.metrics['sharpe_ratio']:.3f}, return={momentum_result.metrics['total_return']:.2%}")


2026-04-16T04:46:27Z [INFO] Starting backtest: strategy=momentum, range=[2024-01-01, 2024-12-31], run_id=fe1807a5
2026-04-16T04:46:27Z [INFO] Loading parquet feed: data/raw/btcusdt_1h.parquet
2026-04-16T04:46:27Z [INFO]   Loaded 8784 bars: 2024-01-01 00:00:00 to 2024-12-31 23:00:00
2026-04-16T04:46:27Z [INFO] Cost model: effective_7bps_per_side (fee=4.0bps + slip=3.0bps = 7.0bps per side)
2026-04-16T04:46:27Z [INFO] Cerebro configured: cash=10000.00, coc=False, coo=False, effective_7bps_per_side, sizer=99%
2026-04-16T04:46:28Z [INFO] Metrics: return=0.0458 sharpe=0.299 maxDD=0.3046 trades=118 win_rate=0.42 PF=1.04
2026-04-16T04:46:28Z [INFO] Trade log saved: /Users/yutianyang/Documents/GitHub/btc-alpha-pipeline/data/results/trades_fe1807a5-f9de-44f9-9016-b72f12410de8.csv (118 trades)
2026-04-16T04:46:28Z [INFO] Ensured 'runs' table exists with Phase 1A schema
2026-04-16T04:46:28Z [INFO] Inserted run fe1807a5-f9de-44f9-9016-b72f12410de8 (strategy: momentum)
2026-04-16T04:46:28Z [INFO] B

,field,value
0,strategy_name,momentum
1,run_id,fe1807a5-f9de-44f9-9016-b72f12410de8
2,range,2024-01-01T00:00:00+00:00 -> 2024-12-31T23:00:...
3,params,"{'lookback_period': 24, 'entry_threshold': 0.0..."
4,trades,118
5,sharpe_ratio,0.299297
6,total_return,0.045812
7,warmup_bars,24
8,effective_start,2024-01-02 00:00:00
9,trade_csv,/Users/yutianyang/Documents/GitHub/btc-alpha-p...


D9 MOMENTUM SUMMARY PASS
Momentum run_id=fe1807a5-f9de-44f9-9016-b72f12410de8, trades=118, Sharpe=0.299, return=4.58%


### D10. End-to-end automated test summary

Run the dedicated D10 test module and leave a compact, human-readable summary of the categories it covers.

In [19]:
import re
import subprocess
import sys

D10_COMMAND = [sys.executable, "-m", "pytest", "tests/test_phase1_pipeline.py", "-q"]
completed = subprocess.run(D10_COMMAND, capture_output=True, text=True)

print(completed.stdout)
if completed.stderr:
    print(completed.stderr)

passed_match = re.search(r"(\d+) passed", completed.stdout)
skipped_match = re.search(r"(\d+) skipped", completed.stdout)
passed_count = int(passed_match.group(1)) if passed_match else None
skipped_count = int(skipped_match.group(1)) if skipped_match else 0

coverage_rows = [
    {"category": "registry fields", "covered": True, "examples": "run_type, strategy_name, strategy_source, fee_model, warmup, Phase 1A NULL fields"},
    {"category": "result structure", "covered": True, "examples": "equity curve, trade CSV, metrics keys"},
    {"category": "trade price semantics", "covered": True, "examples": "first/last trade fill price equals raw OHLCV open"},
    {"category": "signal/fill timing", "covered": True, "examples": "signal time exactly one hour before fill time"},
    {"category": "commission and volume", "covered": True, "examples": "7bps commission and nonzero fill bars"},
]

d10_summary = pd.DataFrame([
    {"field": "command", "value": " ".join(D10_COMMAND)},
    {"field": "returncode", "value": completed.returncode},
    {"field": "passed_count", "value": passed_count},
    {"field": "skipped_count", "value": skipped_count},
])
display(d10_summary)
display(pd.DataFrame(coverage_rows))

assert completed.returncode == 0, "D10 pytest module failed"
assert passed_count is not None and passed_count > 0, "Could not parse passed test count from pytest output"

print(f"D10 END-TO-END TEST PASS: {passed_count} tests passed in tests/test_phase1_pipeline.py")


...........................                                              [100%]
27 passed in 1.09s



,field,value
0,command,/Users/yutianyang/miniconda3/bin/python -m pyt...
1,returncode,0
2,passed_count,27
3,skipped_count,0


,category,covered,examples
0,registry fields,True,"run_type, strategy_name, strategy_source, fee_..."
1,result structure,True,"equity curve, trade CSV, metrics keys"
2,trade price semantics,True,first/last trade fill price equals raw OHLCV open
3,signal/fill timing,True,signal time exactly one hour before fill time
4,commission and volume,True,7bps commission and nonzero fill bars


D10 END-TO-END TEST PASS: 27 tests passed in tests/test_phase1_pipeline.py
